# Player words

Associated chart: the player-words scrolly (what she does vs. what is done to her)

Output files:
`player_words.json` in `src/data/`.

Input files: `data/fox_wnba.csv`, `data/espn.csv`, `data/cbs_sports.csv`, `data/usa_today.csv`, `data/just_womens_sports.csv`, `data/the_athletic.csv`.

Requirements: spaCy + scikit-learn (first run ~20 minutes; parses cached in `notebooks/.cache/`)

`pip install spacy scikit-learn`
`python -m spacy download en_core_web_sm`


## Setup

In [ ]:
import csv, html, json, re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import fighting_words as fw

DATA = Path("../data")
OUT = Path("../src/data/player_words.json")
CACHE = Path(".cache"); CACHE.mkdir(exist_ok=True)

MIN_MENTIONS = 40   # attachments a player needs to get her own column
TOP_N = 30
MIN_COUNT = 1   # the log-odds prior shrinks a one-off word toward zero on its own; no extra floor
MAX_EXAMPLES = 3

# surname -> canonical; "thomas" (Lia) and "white" (the adjective) stay unmatchable
PLAYERS = {}
for canonical, aliases in {
    "Caitlin Clark": ["caitlin clark", "clark"],
    "Angel Reese": ["angel reese", "reese"],
    "Brittney Griner": ["brittney griner", "griner"],
    "Sophie Cunningham": ["sophie cunningham", "cunningham"],
    "DiJonai Carrington": ["dijonai carrington", "carrington", "dijonai"],
    "Paige Bueckers": ["paige bueckers", "bueckers"],
    "A'ja Wilson": ["a'ja wilson", "aja wilson", "a’ja wilson"],
    "Alyssa Thomas": ["alyssa thomas"],
    "Sue Bird": ["sue bird"],
    "Breanna Stewart": ["breanna stewart", "stewie"],
    "Diana Taurasi": ["diana taurasi", "taurasi"],
    "Chennedy Carter": ["chennedy carter", "chennedy"],
    "Aliyah Boston": ["aliyah boston"],
    "Sabrina Ionescu": ["sabrina ionescu", "ionescu"],
    "Napheesa Collier": ["napheesa collier", "collier", "napheesa"],
    "Natasha Cloud": ["natasha cloud"],
    "Kelsey Plum": ["kelsey plum", "plum"],
    "Candace Parker": ["candace parker"],
    "Kelsey Mitchell": ["kelsey mitchell"],
    "Jonquel Jones": ["jonquel jones", "jonquel"],
    "DeWanna Bonner": ["dewanna bonner", "dewanna", "bonner"],
    "Chelsea Gray": ["chelsea gray"],
    "Arike Ogunbowale": ["arike ogunbowale", "ogunbowale", "arike"],
    "Cameron Brink": ["cameron brink"],
    "Nneka Ogwumike": ["nneka ogwumike", "nneka"],
    "Marina Mabrey": ["marina mabrey", "mabrey"],
    "Jewell Loyd": ["jewell loyd", "loyd"],
    "Lexie Hull": ["lexie hull"],
    "Courtney Vandersloot": ["courtney vandersloot", "vandersloot"],
    "Kahleah Copper": ["kahleah copper", "kahleah"],
}.items():
    for alias in aliases:
        PLAYERS[alias] = canonical
NAME_RE = re.compile(r"\b(" + "|".join(sorted(PLAYERS, key=len, reverse=True)) + r")\b", re.I)
# a surname someone else shares (Ryan Clark, Cade Cunningham): a bare surname
# goes to whoever was last named in full before it (notebooks/lookalikes.py)
from lookalikes import bare_is_her
PRONOUN = {"she", "her", "hers"}
HIPCHECK = re.compile(r"\b(hip[- ]?check)(s|ed|ing)?\b", re.I)
# the small model's lemmatiser strips "-ed" one letter too far on a few
# verbs ("hated" -> "hat", "fined" -> "fin"); the token itself says which
LEMMA_FIX = {"hat": "hate", "rat": "rate", "fat": "fate", "fin": "fine", "rehabbe": "rehab", "dat": "date",
             "kneee": "knee", "stressse": "stress"}
# the same slips in a cached parse, renamed after loading it (never real words)
CACHED_LEMMA_FIX = {"kneee": "knee", "stressse": "stress"}
# a pronoun the resolver gave to the wrong player, spotted on the site:
# (player, word, a phrase from its sentence). That attachment doesn't count for
# her, and its example goes too
NOT_ABOUT_HER = [
    ("Alyssa Thomas", "victim", "like she was the victim of some schoolyard bullying"),   # it's Clark
]
def display_lemma(lemma, text=""):
    if lemma.startswith("hipcheck"):
        return "hip-check"
    # only when the token is an inflection ("hated", "hating", "fined"): a
    # literal "hat" or "fin" keeps its lemma
    fixed = LEMMA_FIX.get(lemma)
    if fixed and text.lower() != lemma:
        return fixed
    return lemma
# college coverage, for ordering examples: the site shows a word's first
# example, and the piece is about the pros
COLLEGE = re.compile(
    r"\b(ncaa|college|collegiate|iowa|hawkeyes?|lsu|uconn|huskies|usc|trojans|notre dame|"
    r"south carolina|gamecocks|stanford|ucla|bruins|tcu|final four|march madness|elite eight|"
    r"sweet 16|freshman|sophomore|big ten|big 12|big east|campus|women's college)\b", re.I)

# "how she plays": adjectives on the things a player does, not on the player.
# "Carrington's dirty foul", "committed a reckless play", "known for physical
# defense", "plays dirty". The victim of a foul does not inherit its adjective.
ACT_NOUNS = {"contribution", "contributions", "showing", "play", "foul", "hit", "style", "defense", "defence", "game", "tactic", "elbow",
             "shove", "push", "screen", "contact", "move", "behavior", "behaviour", "approach",
             "basketball", "effort", "intensity", "physicality", "shot", "pass", "shooting",
             "passing", "rebounding", "offense", "offence", "performance", "season", "night",
             "start", "career", "stretch", "run", "minute", "quarter", "half", "outing", "debut",
             # the on-court acts themselves, counted as play words in their own right
             "flagrant", "technical", "ejection", "hipcheck", "hip-check", "punch", "poke", "swipe",
             "jab", "trip", "tackle", "takedown", "block", "steal", "assist", "turnover", "layup",
             "jumper", "three", "three-pointer", "dunk", "bucket", "rebound", "taunt", "celebration",
             "shove", "swing", "grab", "hold", "collision", "hustle", "grit", "toughness", "aggression"}
# an act under one of these isn't hers: "delivered a receipt for a dirty foul"
# answers someone else's foul
ANSWER_NOUNS = {"receipt", "payback", "revenge", "retaliation", "response", "answer", "reply"}
# the acts that count as play words in their own right (the rest of
# ACT_NOUNS only hold an adjective: "a strong season")
PLAY_NOUNS = {
    # the acts themselves count as play words when they carry a judgement: a
    # foul, a shove, her physicality. Neutral act nouns (play, defense, style,
    # effort, approach) do not: there the word is the modifier on them —
    # "fiery and physical defense", "impressive part of her play"
    "foul", "flagrant", "technical", "ejection", "hit", "elbow", "shove", "push",
    "hipcheck", "hip-check", "punch", "poke", "swipe", "jab", "tackle", "contact",
    "screen", "collision", "intensity", "physicality", "hustle", "grit", "toughness", "aggression",
}
# box-score nouns (assist, rebound, shot, three ...) stay in ACT_NOUNS as anchors:
# an adjective on one ("clutch shot") is about her play, the noun itself is not
# verbs that make an act someone else's (she reacts to it or talks about it),
# and the verbs of making contact oneself — from the play audit
RECEIVE_ACT = {"exaggerate", "draw", "absorb", "sell", "embellish", "endure", "suffer", "withstand", "feel", "fake", "flop", "take", "avoid", "survive"}
COMMENT_ACT = {"have", "see", "watch", "complain", "criticize", "decry", "blast", "slam", "address", "discuss", "question",
               "react", "respond", "defend", "talk", "mention", "call", "describe", "label", "dismiss", "deny", "claim",
               # "Thomas accused Fever fans of racist behavior" filed the fans' behaviour
               # as hers
               "accuse", "allege", "blame", "condemn", "denounce", "report", "explain", "acknowledge",
               "recall", "highlight", "say"}
MAKE_ACT = {"make", "initiate", "deliver", "land", "throw", "commit", "put", "lower", "lead"}
PHYSICAL = re.compile(r"\b(foul|flagrant|technical|elbow|shove|court|floor|layup|rim|paint|basket|screen|ejected|whistle|referee|ref|face|eye|head|arm|chest|back|body)\w*", re.I)
METAPHOR = re.compile(r"\b(gut|social media|of words|verbal|playoff|postseason|final push|late push|late-season|standings|campaign)\b", re.I)
# capitalised nouns that are words, not names (everything else capitalised mid-sentence is dropped)
TITLE_WORDS = {"olympian", "olympians", "all-star", "all-stars", "allstar", "famer", "hall-of-famer", "rookie",
               "american", "americans", "christian", "republican", "democrat", "conservative", "liberal"}
# the words that may stand before "game"/"play" and keep it on the court ("playoff game", "post play")
COURT_COMPOUNDS = {"playoff", "postseason", "season", "home", "road", "championship", "finals", "final", "title", "all-star",
                   "post", "perimeter", "transition", "guard", "team", "rookie", "career", "preseason", "exhibition",
                   "elimination", "rivalry", "league", "wnba", "fever", "sky", "aces", "liberty"}
# stat-line abbreviations and units are numbers, not descriptions ("20.5 ppg")
STAT_WORDS = {"ppg", "rpg", "apg", "spg", "bpg", "mpg", "topg", "fg", "fgs", "fga", "fgm", "ft", "fta", "ftm",
              "3pt", "3p", "pct", "percent", "per", "pts", "reb", "rebs", "ast", "asts", "stl", "blk", "min", "mins",
              "points", "rebounds", "assists", "steals", "blocks", "turnovers", "minutes", "possessions"}
# the places a predicate adjective can point at to be about her play
COURT_WORDS = {"field", "floor", "court", "three", "arc", "line", "paint", "perimeter", "range", "glass", "post",
               "rim", "basket", "defense", "defence", "offense", "offence", "ball", "boards", "stripe"}
PLAY_VERBS = {"play", "defend", "compete", "guard", "foul", "hit", "go", "shoot", "rebound"}
# "called Carrington dirty", "described her as a dirty player": a label counts
# as a description of her
LABEL_VERBS = {"call", "label", "consider", "deem", "describe", "brand", "regard", "dub"}

SKIP_VERBS = {"be", "have", "do", "say", "get", "go", "make", "take", "come", "see",
              "know", "think", "want", "look", "give", "use", "find", "tell", "become"}
SKIP_ADJ = {"other", "more", "most", "such", "same", "own", "many", "much", "few",
            "last", "next", "first", "second", "new", "old", "good", "big", "sure",
            "able", "likely", "several", "former", "recent", "about", "no", "only",
            "right", "free", "late", "key", "current", "third", "fourth", "fifth",
            "sixth", "seventh", "eighth", "ninth", "tenth", "due", "least", "front",
            "whole", "full", "high", "low", "long", "short", "early", "late"}

# nouns that say what someone is ("a villain", "the enforcer", "fan favorite")
# count as descriptions; these ones say nothing
SKIP_NOUN = {"player", "one", "part", "member", "person", "people", "thing",
             "name", "case", "example", "time", "year", "game", "season", "way",
             "pick", "point", "points", "team", "guard", "forward", "center",
             "rookie", "veteran", "teammate", "woman", "women", "girl", "man",
             "reason", "specie", "species", "rebound", "stat", "stats", "knee",
             "ankle", "foot", "hamstring", "quad", "groin", "back", "injury",
             "status", "question", "no", "yes", "day", "night", "week", "month"}
# the pieces of the names themselves must not count as descriptions
NAME_TOKENS = {tok for alias in PLAYERS for tok in alias.replace("'", "").split()} | \
              {tok for alias in PLAYERS for tok in alias.split()}

# Fox staples promo lines mid-paragraph; the all-caps run defeats spaCy's splitter
BOILERPLATE = re.compile(r"CLICK HERE[^a-z]*(?=[A-Z][a-z]|$)")

## Corpora

Headline plus full body for every article with one: Fox as-is; the comparison outlets are the piece's usual five: ESPN, CBS Sports, USA Today, Just Women's Sports, The Athletic.


In [ ]:
csv.field_size_limit(10 ** 9)

def article_text(row):
    # the headline is a sentence of its own, ahead of the body
    head = html.unescape((row.get("headline") or "").strip())
    body = BOILERPLATE.sub(" ", html.unescape(row.get("body") or "")).strip()
    # "hip-checked" defeats the parser (Clark comes out as the subject of
    # "was"); joined, "hipchecked" parses as a plain verb. It is written
    # back as hip-check in the counts and the examples.
    head = HIPCHECK.sub(r"hipcheck\2", head)
    body = HIPCHECK.sub(r"hipcheck\2", body)
    if head and not head.endswith((".", "!", "?")):
        head += "."
    return f"{head} {body}".strip()

def fox_articles():
    for row in csv.DictReader((DATA / "fox_wnba.csv").open()):
        text = article_text(row)
        if len(text.split()) > 80:
            yield {"url": row.get("url", ""), "body": text, "date": row.get("date", "")}

import sys as _sys; _sys.path.insert(0, ".")
from outlets import OUTLETS as OUTLET_TABLE
def other_articles():
    for name in [key for key, label, group, *_ in OUTLET_TABLE if group != "fox"]:
        for row in csv.DictReader((DATA / f"{name}.csv").open()):
            outlet = row.get("outlet") or ""
            text = article_text(row)
            if outlet and not outlet.startswith("Fox") and len(text.split()) > 80:
                yield {"url": row.get("url", ""), "body": text, "outlet": outlet, "date": row.get("date", "")}

fox_rows = list(fox_articles())
other_rows = list(other_articles())
len(fox_rows), len(other_rows)

## Extraction

Dependency parse extraction

In [ ]:
def verb_form(token):
    # a hyphenated verb stays whole: "hip-checked" -> "hip-check", not "check"
    if (token.i >= 2 and token.nbor(-1).text == "-"
            and not token.nbor(-1).whitespace_ and not token.nbor(-2).whitespace_
            and token.nbor(-2).is_alpha):
        return token.nbor(-2).text.lower() + "-" + token.lemma_.lower()
    return token.lemma_.lower()


def adjective_form(token):
    # a comparative or superlative keeps its own form: spaCy lemmatises
    # "worst" to "bad" and "best" to "good", which then reads on the chart as a
    # word the example sentence does not contain
    head = token.text.lower() if token.tag_ in {"JJR", "JJS", "RBR", "RBS"} else token.lemma_.lower()
    # rejoin hyphenated compounds: "all-American" not "american"
    parts, cursor = [head], token
    while (cursor.i >= 2 and cursor.nbor(-1).text == "-"
           and not cursor.nbor(-1).whitespace_ and not cursor.nbor(-2).whitespace_):
        cursor = cursor.nbor(-2)
        parts.insert(0, cursor.text.lower())
    return "-".join(parts)


TERMINAL = re.compile(r"[.!?…]['\u2019\"\u201d)\]]*$")

def whole_sentence(sent):
    # spaCy's small model splits inside quotations ("Goldberg said, “Not because
    # she was black" | "but because they didn't believe in the WNBA.”"): a
    # piece with no closing punctuation takes the next piece(s), and one that
    # starts lowercase takes the piece before it, so the sentence reads whole.
    # Returns the text and its token span [start, end).
    doc, start, end = sent.doc, sent.start, sent.end
    text = sent.text.strip()
    while not TERMINAL.search(text) and end < len(doc) and len(text) < 400:
        nxt = doc[end].sent
        if nxt.start != end:
            break
        text, end = text + " " + nxt.text.strip(), nxt.end
    while text[:1].islower() and start > 0 and len(text) < 400:
        prev = doc[start - 1].sent
        if prev.end != start:
            break
        text, start = prev.text.strip() + " " + text, prev.start
    return text, start, end

MARK = "\x01"   # rides along with the word through the trimming below, then comes out

def window_example(sent, token, budget=900):
    # the stored example is the sentence with the one before and the one
    # after it: a reader can see who "she" is, so a pronoun the coreference
    # resolver attached to the wrong player shows up as wrong on the card.
    # Returns the text and where the word starts in it (the same word can
    # appear twice: "a black player ... a black eye")
    doc = sent.doc
    text, a, b = whole_sentence(sent)
    pos = token.idx - doc[a].idx
    if a > 0:
        prev, pa, pb = whole_sentence(doc[a - 1].sent)
        if pb == a:
            text, pos = prev + " " + text, pos + len(prev) + 1
    if b < len(doc):
        nxt, na, nb_ = whole_sentence(doc[b].sent)
        if na == b:
            text = text + " " + nxt
    raw = text[:pos] + MARK + text[pos:]
    if len(raw) <= budget:
        piece, pre, post = raw, False, False
    else:
        start = max(0, pos - budget // 2)
        end = min(len(raw), start + budget)
        start = max(0, min(start, end - budget))
        piece, pre, post = raw[start:end], start > 0, end < len(raw)
        if pre:
            piece = piece.split(" ", 1)[-1]
        if post:
            piece = piece.rsplit(" ", 1)[0]
    text = " ".join(html.unescape(piece).split())
    text = re.sub(r"\b[Hh]ipcheck", lambda m: m.group(0)[0] + "ip-check", text)
    text = ("…" if pre else "") + text + ("…" if post else "")
    at = text.find(MARK)
    return text.replace(MARK, ""), (at if at >= 0 else None)



# two-word phrases per relation: the attached word and its
# nearest partner in the parse. A verb takes its particle ("foul out"), else
# its object ("draw foul"), else a complement or a telling adverb; an
# adjective takes a telling adverb, else the noun it sits on ("cheap shot");
# a noun takes its adjective or compound ("biggest star"); an adverb takes its
# verb ("shoot efficiently"). Intensifiers and time words are not partners.
WEAK_ADV = {"very", "so", "too", "really", "more", "most", "as", "also", "even", "just", "quite",
            "pretty", "then", "now", "still", "again", "ever", "already", "only", "not", "never",
            "back", "how", "when", "where", "why", "here", "there", "all", "much", "well", "yet"}

def _partner_word(t):
    if t.pos_ == "VERB":
        return verb_form(t)
    if t.pos_ == "ADJ":
        return adjective_form(t)
    return t.lemma_.lower()

def _partner_ok(t):
    return (t.is_alpha and not t.is_stop and t.pos_ in {"NOUN", "VERB", "ADJ", "ADV"}
            and t.text.lower() not in NAME_TOKENS
            and not (t.text[:1].isupper() and t.i != t.sent.start))

def two_word(tok, lemma):
    kids = list(tok.children)
    partner = None
    if tok.pos_ in {"VERB", "AUX"}:
        for ch in kids:
            if ch.dep_ == "prt" and ch.is_alpha:
                return f"{lemma} {ch.text.lower()}"
        for deps in ({"dobj", "obj"}, {"acomp"}):
            partner = next((ch for ch in kids if ch.dep_ in deps and _partner_ok(ch)), None)
            if partner is not None:
                break
        if partner is None:
            partner = next((ch for ch in kids if ch.dep_ == "advmod" and ch.lemma_.lower() not in WEAK_ADV
                            and _partner_ok(ch)), None)
    elif tok.pos_ == "ADJ":
        partner = next((ch for ch in kids if ch.dep_ == "advmod" and ch.lemma_.lower() not in WEAK_ADV
                        and _partner_ok(ch)), None)
        if partner is None and tok.dep_ == "amod" and tok.head.pos_ == "NOUN" and _partner_ok(tok.head):
            partner = tok.head
    elif tok.pos_ == "NOUN":
        partner = next((ch for ch in kids if ch.dep_ in {"amod", "compound"} and _partner_ok(ch)), None)
    elif tok.pos_ == "ADV":
        if tok.head.pos_ == "VERB" and _partner_ok(tok.head):
            partner = tok.head
    if partner is None:
        return None
    word = _partner_word(partner)
    return f"{word} {lemma}" if partner.i < tok.i else f"{lemma} {word}"

# a describing or play word inside someone's quotation, hanging on something
# else in that quotation, is not about her: "She described the Fever faithful
# as "the nastiest fans"". Scare quotes on her ("labeled a
# "villain"") hang on a verb outside the quote and still count.
QUOTE_MARKS = '"“”'

def quote_spans(sent):
    # the sentence's quoted stretches as doc character ranges [a, b); marks are
    # paired in order, and one left unpaired runs to the sentence's edge
    text, base = sent.text, sent.start_char
    marks = [i for i, ch in enumerate(text) if ch in QUOTE_MARKS]
    spans = []
    if marks and len(marks) % 2 == 1:
        if text[marks[0]] == "”":
            spans.append((base, base + marks[0]))
            marks = marks[1:]
        else:
            spans.append((base + marks[-1], base + len(text)))
            marks = marks[:-1]
    for a, b in zip(marks[0::2], marks[1::2]):
        spans.append((base + a, base + b + 1))
    return spans

def quoted_away(word_token, her_token):
    # the word and what it hangs on sit inside one quotation, and she is
    # outside it: the quote is about something else ("the nastiest fans")
    for a, b in quote_spans(word_token.sent):
        if a <= word_token.idx < b and a <= word_token.head.idx < b and not (a <= her_token.idx < b):
            return True
    return False


def extract(nlp, rows):
    found = defaultdict(lambda: defaultdict(Counter))
    examples = defaultdict(list)
    by_outlet = defaultdict(lambda: defaultdict(lambda: defaultdict(Counter)))
    by_outlet_examples = defaultdict(lambda: defaultdict(list))   # outlet -> key -> up to 2 sentences, for the single-outlet pills
    seen, texts, urls = set(), [], []
    outlet_of = {}
    period_of = {}
    for row in rows:
        if row["url"] in seen:
            continue
        seen.add(row["url"])
        texts.append(row["body"])
        urls.append(row["url"])
        outlet_of[row["url"]] = row.get("outlet", "Fox News")
        # the era, as a pseudo-outlet key ("@pre" 2019-23, "@clark" 2024-26) beside
        # the real outlets, for the before/after-Clark comparison
        period_of[row["url"]] = "@pre" if (row.get("date") or "")[:4] < "2024" else "@clark"
    print(f"{len(texts)} articles", flush=True)

    for n, (doc, url) in enumerate(zip(nlp.pipe(texts, batch_size=20), urls), 1):
        # who each pronoun stands for, by coreference (COREF, cached)
        mentions = COREF.get(url, [])
        def by_coref(token):
            # (player, confidence); an old cache row has no confidence
            for a, b, who, *conf in mentions:
                if a <= token.idx < b:
                    return who, (conf[0] if conf else None)
            return None, None
        if n % 2000 == 0:
            print(f"  parsed {n}/{len(texts)}", flush=True)
        for sent in doc.sents:
            # a name can be several tokens ("Alyssa Thomas", "A'ja Wilson"): the
            # match is made on the sentence text and pinned to the span's last
            # token, which is the one the parse attaches to the verb
            named_at = {}
            names = []
            for m in NAME_RE.finditer(sent.text):
                who = PLAYERS[m.group(1).lower()]
                if " " not in m.group(1) and not bare_is_her(who, doc.text, sent.start_char + m.start(), sent.start_char):
                    continue
                names.append(who)
                span = sent.char_span(m.start(), m.end(), alignment_mode="expand")
                if span is not None:
                    named_at[span[-1].i] = who
            for token in sent:
                who, conf, pron = None, 1.0, None
                low = token.text.lower()
                if low in PRONOUN:
                    who, conf = by_coref(token)
                    pron = low
                elif token.i in named_at:
                    who = named_at[token.i]
                if not who:
                    continue
                # what an example records besides its text: the pronoun it
                # hangs on (if any) and the resolver's confidence, for the audit
                def stamp(entry):
                    entry["p"] = conf
                    if pron:
                        entry["pr"] = pron
                    return entry

                def note(relation, lemma, word_token):
                    if relation in ("describe", "play") and quoted_away(word_token, token):
                        return
                    # a capitalised noun mid-sentence is a name the roster does
                    # not know ("Kristi Toliver", "Delle Donne"), not a word about
                    # her; titles that are words stay ("Olympian", "All-Star", "MVP")
                    wt = word_token.text
                    if word_token.pos_ in {"NOUN", "PROPN"} and wt[:1].isupper() and not wt.isupper() \
                            and word_token.i != word_token.sent.start and wt.lower() not in TITLE_WORDS:
                        return
                    if word_token.text.lower() in STAT_WORDS or lemma.split("|")[0] in STAT_WORDS:
                        return
                    lemma = display_lemma(lemma, word_token.text)
                    # a negated word is a different word: "wasn't overly efficient"
                    # is filed as not-efficient, "didn't foul" as not-foul. The
                    # negation sits on the verb, or on the copula above a predicate
                    tok = word_token
                    negated = any(ch.dep_ == "neg" for ch in tok.children) or \
                        (tok.dep_ in {"acomp", "attr", "oprd", "conj"} and any(ch.dep_ == "neg" for ch in tok.head.children)) or \
                        any(ch.dep_ == "advmod" and ch.lemma_.lower() in {"never", "hardly", "barely", "rarely"} for ch in tok.children)
                    if negated and not lemma.startswith("not-"):
                        lemma = "not-" + lemma
                    def _record(relation, lemma):
                        found[who][relation][lemma] += 1
                        by_outlet[outlet_of.get(url, "Fox News")][who][relation][lemma] += 1
                        by_outlet[period_of.get(url, "@clark")][who][relation][lemma] += 1
                        own = by_outlet_examples[outlet_of.get(url, "Fox News")][f"{who}|{relation}|{lemma}"]
                        if len(own) < 2:
                            own_line, own_at = window_example(sent, word_token)
                            if all(own_line != e["t"] for e in own):
                                own.append(stamp({"t": own_line, "w": word_token.text.lower(), "a": own_at, "o": outlet_of.get(url, "Fox News")}))
                        key = f"{who}|{relation}|{lemma}"
                        kept = examples[key]
                        # the first example is the one the site shows: a WNBA-era
                        # sentence goes ahead of a college one unless the word
                        # itself is a college word
                        if len(kept) < MAX_EXAMPLES or (kept and COLLEGE.search(kept[0]["t"])):
                            line, at = window_example(sent, word_token)
                            if all(line != e["t"] for e in kept):
                                entry = stamp({"t": line, "w": word_token.text.lower(), "a": at, "o": outlet_of.get(url, "Fox News")})
                                pro = not COLLEGE.search(line) and not COLLEGE.search(lemma)
                                if pro and kept and COLLEGE.search(kept[0]["t"]):
                                    kept.insert(0, entry)
                                    del kept[MAX_EXAMPLES:]
                                elif len(kept) < MAX_EXAMPLES:
                                    kept.append(entry)
                    _record(relation, lemma)
                    phrase = two_word(word_token, lemma)
                    if phrase:
                        _record(relation + "2", phrase)

                # a coordinated name ("Clark and Cunningham were fouled") takes the
                # first conjunct's place in the parse
                anchor = token
                while anchor.dep_ == "conj" and anchor.head is not anchor:
                    anchor = anchor.head
                dep, head = anchor.dep_, anchor.head
                usable = lambda verb: verb.pos_ == "VERB" and verb.lemma_.lower() not in SKIP_VERBS
                # "Clark scored 20 but wasn't efficient", "scored 20 and grabbed 10":
                # the verbs coordinated with hers have her as their subject too
                # ... unless that verb has a subject of its own the parser hung
                # elsewhere: "Carrington is Black and Cunningham is white" files
                # Cunningham under "Black", so a name or noun just before the verb
                # (past its auxiliaries) is taken as its subject
                def own_subject(verb):
                    prev = verb.i - 1
                    while prev > 0 and verb.doc[prev].pos_ in {"AUX", "PART"}:
                        prev -= 1
                    return prev >= 0 and verb.doc[prev].pos_ in {"PROPN", "NOUN", "PRON"}
                co_heads = [c for c in head.children if c.dep_ == "conj" and c.pos_ in {"VERB", "AUX"}
                            and not own_subject(c)] \
                    if dep in {"nsubj", "csubj"} and head.pos_ in {"VERB", "AUX"} else []

                # a relative clause on the name: "Cunningham, who punched Clark"
                for clause in token.children:
                    if clause.dep_ == "relcl" and usable(clause):
                        for rel in clause.children:
                            if rel.tag_ in {"WP", "WDT"} or rel.text.lower() in {"who", "whom", "that", "which"}:
                                if rel.dep_ in {"nsubj", "csubj"}:
                                    note("subject", verb_form(clause), clause)
                                elif rel.dep_ in {"nsubjpass", "dobj", "obj"}:
                                    note("object", verb_form(clause), clause)

                if dep == "pobj" and head.pos_ == "ADP":
                    verb = head.head
                    if head.dep_ == "agent" and usable(verb):
                        # a passive's doer: "punched in the throat by Alyssa Thomas"
                        note("subject", verb_form(verb), verb)
                    elif head.dep_ == "prep" and usable(verb):
                        # the verb's target through a preposition: "took a swipe at
                        # Cunningham", "went after Cunningham"
                        note("object", verb_form(verb), verb)
                    # "performances from Aliyah Boston": go on to the play rules below
                    if not (head.lemma_.lower() in {"from", "by"} and head.head.pos_ in {"NOUN", "VERB"}
                            and (head.head.lemma_.lower() in ACT_NOUNS or head.head.text.lower() in ACT_NOUNS)):
                        continue
                if head.pos_ in {"VERB", "AUX"} and head.lemma_.lower() not in SKIP_VERBS:
                    if dep in {"nsubj", "csubj", "agent"}:
                        note("subject", verb_form(head), head)
                    elif dep in {"dobj", "obj", "nsubjpass", "pobj"}:
                        note("object", verb_form(head), head)
                for h2 in co_heads:
                    if h2.lemma_.lower() not in SKIP_VERBS:
                        note("subject", verb_form(h2), h2)

                # JJ/JJR/JJS are real adjectives; VBN is a verb wearing an adjective's hat
                def adjectives_on(noun):
                    for child in noun.children:
                        if child.tag_.startswith("JJ") and child.dep_ == "amod" \
                                and child.lemma_.lower() not in SKIP_ADJ:
                            note("describe", adjective_form(child), child)

                def noun_describes(noun):
                    # "a villain", "the enforcer", "fan favorite": the noun itself
                    lemma = noun.lemma_.lower()
                    if noun.pos_ == "NOUN" and lemma not in SKIP_NOUN and lemma.isalpha() \
                            and noun.text.lower() not in NAME_TOKENS:
                        note("describe", lemma, noun)
                    adjectives_on(noun)

                # --- how she plays: adjectives on her own acts, adverbs on her verbs ---
                def play_adjectives(noun, verb=""):
                    # "hard fouls from opponents", "a hit by Reese": someone else's act —
                    # unless the one after "from"/"by" is her ("solid performances
                    # from Aliyah Boston")
                    for c in noun.children:
                        if c.dep_ in {"prep", "agent"} and c.lemma_.lower() in {"from", "by"}:
                            if not any(o.i == token.i or (o.i in named_at and named_at[o.i] == who) for o in c.children):
                                return
                    # "Clark exaggerated the contact", "drew a foul", "has had enough of
                    # the dirty fouls on Clark": the act is someone else's, she only
                    # reacts to it or talks about it
                    if verb in RECEIVE_ACT or verb in COMMENT_ACT:
                        return
                    lemma = noun.lemma_.lower()
                    # "her social media game", "mind games", "the waiting game": a
                    # game or play with a non-basketball word in front is not on the court
                    if lemma in {"game", "play", "night", "performance", "showing"} and any(
                            ch.dep_ == "compound" and ch.lemma_.lower() not in COURT_COMPOUNDS for ch in noun.children):
                        return
                    # contact and a hit are hers only when she makes them: her own
                    # ("Carrington's hit"), or a verb of doing ("initiated contact")
                    if lemma in {"contact", "hit"} and verb and verb not in MAKE_ACT:
                        return
                    # push, jab, punch: physical only beside the court; otherwise a
                    # playoff push, a verbal jab, a gut punch
                    if lemma in {"push", "jab", "punch"} and (not PHYSICAL.search(noun.sent.text) or METAPHOR.search(noun.sent.text)):
                        return
                    # "flagrant 2 foul", "flagrant foul 2": the parser hangs the
                    # adjective on the number, so look one level down too
                    # the act itself: "committed a foul", "was assessed a flagrant 2",
                    # "set a screen" — every word attached to what she did on the court
                    if noun.lemma_.lower() in PLAY_NOUNS:
                        note("play", noun.lemma_.lower() + "|" + noun.lemma_.lower() + "|" + verb, noun)
                    def note_adj(adj):
                        if adj.tag_.startswith("JJ") and adj.lemma_.lower() not in SKIP_ADJ:
                            act = noun if (noun.lemma_.lower() in ACT_NOUNS or noun.text.lower() in ACT_NOUNS) else (act_compound(noun) or noun)
                            note("play", adjective_form(adj) + "|" + (act.text.lower() if act.pos_ == "VERB" else act.lemma_.lower()) + "|" + verb, adj)
                        # "fiery and physical defense": the adjective joined by "and" too
                        for more in adj.children:
                            if more.dep_ == "conj" and more.tag_.startswith("JJ"):
                                note_adj(more)
                    holders = [noun] + [c for c in noun.children if c.dep_ in {"nummod", "compound"} or c.pos_ == "NUM"]
                    for holder in holders:
                        for child in holder.children:
                            if child.dep_ == "amod":
                                note_adj(child)
                    # "the most impressive part of her play": the adjective sits on
                    # the noun above the act ("part of", "aspect of", "brand of")
                    if noun.dep_ == "pobj" and noun.head.lemma_.lower() == "of" and noun.head.head.pos_ == "NOUN" \
                            and noun.head.head.lemma_.lower() in {"part", "aspect", "element", "piece", "version", "brand",
                                                                  "kind", "level", "display", "side", "feature", "example"}:
                        for child in noun.head.head.children:
                            if child.dep_ == "amod":
                                note_adj(child)
                    # "just as electric as her actual game": the comparison says her
                    # game is electric too — the adjective the "as … as" hangs on
                    if noun.dep_ == "pobj" and noun.head.lemma_.lower() == "as" and noun.head.head.pos_ == "ADJ":
                        note_adj(noun.head.head)
                    # "her defense was elite": a predicate adjective on the act
                    if noun.dep_ in {"nsubj", "nsubjpass"} and noun.head.pos_ in {"AUX", "VERB"}:
                        for child in noun.head.children:
                            if child.dep_ == "acomp":
                                note_adj(child)

                # an act noun by lemma, or by its surface form when the parser
                # tags a gerund as a verb ("on an incredibly efficient 15-for-19
                # shooting", "her rebounding"): the word is the same act
                def is_act(tok):
                    return (tok.pos_ == "NOUN" and tok.lemma_.lower() in ACT_NOUNS) or \
                           (tok.pos_ in {"NOUN", "VERB"} and tok.text.lower() in ACT_NOUNS) or \
                           act_compound(tok) is not None

                # "her ugly shooting streak", "a scoring slump", "poor shot selection":
                # the act is the compound in front, and the adjective is about it
                #
                def act_compound(tok):
                    if tok.pos_ != "NOUN":
                        return None
                    return next((c for c in tok.children if c.dep_ == "compound"
                                 and (c.lemma_.lower() in ACT_NOUNS or c.text.lower() in ACT_NOUNS)), None)

                def act_nouns_under(node):
                    # the act noun itself, or one preposition down: "history of dirty play"
                    # (not under a receipt, payback ...: that answers someone else's act)
                    if is_act(node):
                        yield node
                    if node.lemma_.lower() in ANSWER_NOUNS:
                        return
                    for child in node.children:
                        if child.dep_ == "prep":
                            for obj in child.children:
                                if obj.dep_ == "pobj" and is_act(obj):
                                    yield obj

                # "Carrington's dirty foul", "her physical style"
                if dep == "poss" and is_act(head):
                    play_adjectives(head)
                # "solid performances from Aliyah Boston", "huge games from Boston",
                # "great contributions from her": the act noun above "from" is hers
                if dep == "pobj" and head.lemma_.lower() in {"from", "by"} and is_act(head.head):
                    play_adjectives(head.head)
                # the act under a verb she is the subject of — active ("committed a
                # dirty play", "has a history of dirty play", "is known for physical
                # play") or passive ("was assessed a flagrant foul", "was ejected
                # for a hard foul", "hit with a flagrant 2")
                for h in ([head] + co_heads if dep in {"nsubj", "csubj", "nsubjpass"} and head.pos_ in {"VERB", "AUX"} else []):
                    for child in h.children:
                        if child.dep_ in {"dobj", "attr", "prep", "oprd"}:
                            # "was the victim of hard fouls": not her play
                            if child.dep_ == "attr" and child.lemma_.lower() in {"victim", "target", "recipient", "subject"}:
                                continue
                            for noun in act_nouns_under(child):
                                play_adjectives(noun, h.lemma_.lower())
                        # "plays dirty", "played physical"
                        if dep != "nsubjpass" and child.dep_ in {"acomp", "advmod"} \
                                and (child.tag_.startswith("JJ") or child.tag_ == "RB") \
                                and h.lemma_.lower() in PLAY_VERBS \
                                and child.lemma_.lower() not in SKIP_ADJ | {"not", "also", "just", "still", "again", "even", "so", "very", "then", "well", "too", "there", "here", "now", "back", "up", "down", "out", "off", "on", "in", "away", "over", "through", "along", "around", "forward", "ahead", "home",
                                        # adverbs of time and stance, not of manner
                                        "recently", "actually", "obviously", "simply", "prior", "currently", "previously", "eventually", "finally", "already", "mostly", "really", "probably", "clearly", "apparently", "reportedly", "allegedly", "basically", "certainly", "definitely", "literally", "ultimately", "usually", "often", "sometimes", "always", "never", "later", "earlier", "first", "last", "once", "twice", "again", "yet", "ever", "much", "more", "most", "less", "far", "long", "enough", "instead", "however", "otherwise", "together", "alone", "elsewhere", "especially", "particularly", "generally", "typically", "regularly", "consistently", "immediately", "quickly", "soon", "late", "early", "today", "tonight", "yesterday", "tomorrow"}:
                            note("play", adjective_form(child) + "|" + h.lemma_.lower() + "|" + h.lemma_.lower(), child)

                # the act inside a relative clause on her name is hers too:
                # "Cunningham, who committed a hard foul", "who was assessed a
                # flagrant 1 foul"
                for clause in token.children:
                    if clause.dep_ == "relcl" and usable(clause):
                        if any(rel.dep_ in {"nsubj", "csubj", "nsubjpass"} and
                               (rel.tag_ in {"WP", "WDT"} or rel.text.lower() in {"who", "whom", "that", "which"})
                               for rel in clause.children):
                            for part in clause.children:
                                if part.dep_ in {"dobj", "attr", "prep", "oprd"}:
                                    for noun in act_nouns_under(part):
                                        play_adjectives(noun, clause.lemma_.lower())

                # --- labels others put on her: "called Carrington dirty",
                # "described her as a dirty player" ---
                if dep in {"dobj", "nsubjpass"} and head.lemma_.lower() in LABEL_VERBS:
                    for child in head.children:
                        if child.dep_ in {"oprd", "acomp", "ccomp", "xcomp"}:
                            if child.tag_.startswith("JJ") and child.lemma_.lower() not in SKIP_ADJ:
                                note("describe", adjective_form(child), child)
                            elif child.pos_ == "NOUN":
                                noun_describes(child)
                        if child.dep_ == "prep" and child.lemma_.lower() == "as":
                            for obj in child.children:
                                if obj.dep_ == "pobj" and obj.pos_ == "NOUN":
                                    noun_describes(obj)
                                elif obj.dep_ == "pobj" and obj.tag_.startswith("JJ"):
                                    note("describe", adjective_form(obj), obj)

                adjectives_on(token)
                for child in token.children:
                    # "outspoken Fever guard Sophie Cunningham", "Cunningham, the
                    # Fever's fiery guard": the noun beside the name, and its adjectives
                    if child.dep_ in {"compound", "appos", "nmod"} and child.pos_ == "NOUN":
                        # a headline the parser read as one long noun phrase ("Fans
                        # torch Chennedy Carter for …": the name is the sentence's
                        # root and the verb became a noun) has no description in
                        # front of the name; nor does a plural ("fans", "critics")
                        if child.dep_ == "compound" and (token.dep_ == "ROOT" or child.tag_ in {"NNS", "NNPS"}):
                            continue
                        noun_describes(child)
                        # "the 'enforcer' teammate of Clark": a compound on that noun
                        for part in child.children:
                            if part.dep_ == "compound" and part.pos_ in {"NOUN", "PROPN"}:
                                noun_describes(part)

                # --- what she is called: a role noun reached through "as" ("stepping
                # up as the team's enforcer", "known as Clark's enforcer", "showing
                # herself as the enforcer"), through a role word ("earned the
                # nickname 'the enforcer'", "the title of enforcer", "her role as
                # Clark's enforcer", "Cunningham's label: 'The Enforcer'") ---
                ROLE_WORDS = {"nickname", "title", "label", "role", "reputation", "moniker", "tag", "name"}
                def role_nouns_under(node):
                    for part in node.children:
                        if part.dep_ == "prep" and part.lemma_.lower() in {"as", "of"}:
                            for obj in part.children:
                                if obj.dep_ == "pobj" and obj.pos_ in {"NOUN", "PROPN"}:
                                    yield obj
                        if part.dep_ == "appos" and part.pos_ in {"NOUN", "PROPN"}:
                            yield part
                if dep in {"nsubj", "csubj", "nsubjpass", "poss"} and head.pos_ in {"VERB", "AUX", "NOUN"}:
                    # "as the enforcer" hanging on her verb, or on a verb one clause
                    # down ("was congratulated for stepping up as the enforcer")
                    verbs = [head] + [c for c in head.children if c.dep_ in {"prep", "xcomp", "advcl"} for c in ([c] + list(c.children)) if c.pos_ == "VERB"]
                    for verb in verbs:
                        for part in verb.children:
                            if part.dep_ == "prep" and part.lemma_.lower() == "as":
                                for obj in part.children:
                                    if obj.dep_ == "pobj" and obj.pos_ in {"NOUN", "PROPN"}:
                                        noun_describes(obj)
                            # "earned the nickname 'the enforcer'", "took on the role of enforcer"
                            if part.dep_ in {"dobj", "attr", "pobj"} and part.lemma_.lower() in ROLE_WORDS:
                                for noun in role_nouns_under(part):
                                    noun_describes(noun)
                            if part.dep_ == "prep":
                                for obj in part.children:
                                    if obj.dep_ == "pobj" and obj.lemma_.lower() in ROLE_WORDS:
                                        for noun in role_nouns_under(obj):
                                            noun_describes(noun)
                    # "Cunningham's label: 'The Enforcer'", "her role as Clark's enforcer"
                    if dep == "poss" and head.lemma_.lower() in ROLE_WORDS:
                        for noun in role_nouns_under(head):
                            noun_describes(noun)
                # a coordinated name takes the appositive only when it has no
                # title of its own: "the fan favorites, Clark and Reese" gives
                # both "favorite", but in "coach Derek Fisher and Sky star
                # Candace Parker" Parker is the star, not the coach
                own_title = token is not anchor and any(
                    ch.dep_ == "compound" and ch.pos_ == "NOUN" for ch in token.children)
                if dep == "appos" and head.pos_ == "NOUN" and not own_title:
                    # "the fan favorite, Sophie Cunningham"
                    noun_describes(head)
                for h in ([head] + co_heads if dep in {"nsubj", "nsubjpass"} else []):
                  if h.pos_ != "AUX":
                    continue
                  if True:
                    for child in h.children:
                        if child.dep_ in {"acomp", "attr"}:
                            if child.tag_.startswith("JJ"):
                                if child.lemma_.lower() not in SKIP_ADJ:
                                    # "efficient from the field", "great on the perimeter": a
                                    # predicate adjective tied to the court describes her play
                                    on_court = any(pr.dep_ == "prep" and any(o.dep_ == "pobj" and o.lemma_.lower() in COURT_WORDS for o in pr.children)
                                                   for pr in child.children) or \
                                        any(ad.dep_ == "advmod" and ad.lemma_.lower() in {"defensively", "offensively"} for ad in child.children)
                                    note("play" if on_court else "describe", adjective_form(child), child)
                            elif child.pos_ == "NOUN":
                                # "Cunningham is a villain" — but a plural can't be one
                                # player unless she's one of a pair ("Clark and Boston
                                # are teammates"): "Copper is a free agent, as are
                                # married Sky teammates Vandersloot and Quigley"
                                paired = token is not anchor or any(c.dep_ == "conj" for c in anchor.children)
                                if child.tag_ == "NNS" and not paired:
                                    continue
                                noun_describes(child)
    return found, dict(examples), by_outlet, {o: dict(v) for o, v in by_outlet_examples.items()}

In [ ]:
# Pronouns are resolved by coreference (fastcoref's LingMess model, the more
# accurate of its two), not by the last name seen: a cluster that contains
# exactly one roster player's name hands that player every pronoun in it.
# Each resolved mention carries the model's confidence (the sigmoid of its
# strongest link to one of the player's own name mentions), so the least
# certain examples can be sent for audit (notebooks/example_audit.py).
# Cached per article (url + text hash), so a rerun only resolves what
# changed. An hour or two the first time.
import gc, hashlib, math, torch, psutil
import lingmess_mps            # lets LingMess run on Apple's GPU (notebooks/lingmess_mps.py)
from fastcoref import LingMessCoref

COREF_CACHE = CACHE / "coref_lingmess.jsonl"
COREF_MAX_CHARS = 20000

def coref_key(row):
    return row["url"] + ":" + hashlib.md5(row["body"].encode("utf-8")).hexdigest()[:10]

def resolve_clusters(text, clusters, pred=None):
    """[[start, end, player, confidence], ...] for every mention in a cluster
    that names exactly one roster player: the pronouns, and short noun phrases
    like 'the rookie' — anything that is not itself another person's name."""
    out = []
    for cluster in clusters:
        players = set()
        cluster = [span for span in cluster if span]   # fastcoref returns None for a span it cannot place
        for a, b in cluster:
            m = NAME_RE.search(text[a:b])
            if m and (" " in m.group(1)
                      or bare_is_her(PLAYERS[m.group(1).lower()], text, a + m.start())):
                players.add(PLAYERS[m.group(1).lower()])
        if len(players) != 1:
            continue
        who = players.pop()
        names = [(a, b) for a, b in cluster if NAME_RE.search(text[a:b])]
        for a, b in cluster:
            span = text[a:b]
            if NAME_RE.search(span):
                continue          # the name itself: matched directly, not by coreference
            if any(ch.isupper() for ch in span[1:]) and span.lower() not in PRONOUN:
                continue          # a capitalised phrase: some other name, leave it
            conf = None
            if pred is not None:
                best = None
                for na, nb_ in names:
                    try:
                        logit = float(pred.get_logit((a, b), (na, nb_)))
                    except Exception:
                        continue
                    best = logit if best is None else max(best, logit)
                if best is not None:
                    conf = round(1 / (1 + math.exp(-best)), 3)
            out.append([a, b, who, conf])
    return out

def coref_map(rows_by_name):
    """{url: [[start, end, player], ...]} for every article, from the cache
    where the text is unchanged and from the model for the rest."""
    done = {}
    if COREF_CACHE.exists():
        for line in COREF_CACHE.open():
            try:
                r = json.loads(line)
                done[r["k"]] = r["m"]
            except Exception:
                pass
    rows = [r for rows in rows_by_name for r in rows]
    todo = [r for r in rows if coref_key(r) not in done]
    print(f"coref: {len(rows):,} articles · {len(rows) - len(todo):,} cached · {len(todo):,} to resolve", flush=True)
    if todo:
        device = "mps" if torch.backends.mps.is_available() else "cpu"
        model = LingMessCoref(device=device)
        if device == "mps":
            # past half the Mac's memory the GPU raises an error, rather than
            # pushing everything into swap and freezing the machine
            torch.mps.set_per_process_memory_fraction(0.5)
        # LingMess scores every pair of mentions, so memory grows with the
        # square of an article's length: short articles first, a few at a time,
        # and the long ones last (and fewer per batch)
        todo.sort(key=lambda r: len(r["body"]))
        proc = psutil.Process()
        with COREF_CACHE.open("a") as out:
            BATCH = 50
            for i in range(0, len(todo), BATCH):
                chunk = todo[i:i + BATCH]
                texts = [r["body"][:COREF_MAX_CHARS] for r in chunk]
                preds = model.predict(texts=texts, max_tokens_in_batch=4000)
                for r, pred in zip(chunk, preds):
                    mentions = resolve_clusters(r["body"], pred.get_clusters(as_strings=False), pred)
                    done[coref_key(r)] = mentions
                    out.write(json.dumps({"k": coref_key(r), "m": mentions}) + "\n")
                out.flush()
                del preds
                gc.collect()
                if device == "mps":
                    torch.mps.empty_cache()
                rss = proc.memory_info().rss / 1e9
                print(f"  {min(i + BATCH, len(todo)):,}/{len(todo):,} · {rss:.1f} GB", flush=True)
                if rss > 12:
                    # everything so far is in the cache: rerun the notebook (a
                    # fresh kernel) and it carries on from here. An error, not a
                    # break, so the parse never runs on half-resolved pronouns
                    raise MemoryError("coref: the process is using over 12 GB; rerun the notebook to continue from the cache")
    return {r["url"]: done.get(coref_key(r), []) for r in rows}

COREF = coref_map([fox_rows, other_rows])
print(f"{sum(1 for m in COREF.values() if m):,} articles with resolved pronouns")
# how sure the resolver is, to set the audit threshold (cell 17: AUDIT_THRESH)
_conf = sorted(m[3] for ms in COREF.values() for m in ms if len(m) > 3 and m[3] is not None)
if _conf:
    q = lambda f: _conf[int(f * (len(_conf) - 1))]
    print(f"confidence over {len(_conf):,} resolved mentions: 10% {q(.1):.2f} · 25% {q(.25):.2f} · median {q(.5):.2f} · "
          f"below 0.85: {sum(c < 0.85 for c in _conf) / len(_conf):.0%}")


In [ ]:
def cached_parse(name, rows):
    cache = CACHE / f"{name}.json"
    if cache.exists():
        raw = json.loads(cache.read_text())
        if "by_outlet_examples" in raw:
            print(f"{name}: reusing {cache} (delete it to re-parse)")
            return ({p: {r: Counter(w) for r, w in b.items()} for p, b in raw["counts"].items()},
                    raw["examples"],
                    {o: {p: {r: Counter(w) for r, w in b.items()} for p, b in store.items()}
                     for o, store in raw["by_outlet"].items()},
                    raw["by_outlet_examples"])
        print(f"{name}: cache predates the per-outlet examples, re-parsing")
    import spacy
    nlp = spacy.load("en_core_web_sm", disable=["ner"])
    found, examples, by_outlet, by_outlet_examples = extract(nlp, rows)
    cache.write_text(json.dumps({
        "counts": {p: {r: dict(w) for r, w in b.items()} for p, b in found.items()},
        "examples": examples,
        "by_outlet": {o: {p: {r: dict(w) for r, w in b.items()} for p, b in store.items()}
                      for o, store in by_outlet.items()},
        "by_outlet_examples": by_outlet_examples}))
    return found, examples, by_outlet, by_outlet_examples

found, examples, fox_by_outlet, _ = cached_parse("fox_parsed", fox_rows)
other_found, other_examples, other_by_outlet, other_by_outlet_examples = cached_parse("others_parsed", other_rows)

def fixed_lemma(lemma):
    return " ".join(CACHED_LEMMA_FIX.get(part, part) for part in lemma.split(" "))

def rename_lemmas(stores, example_stores):
    # stores: {player: {relation: Counter}}; example_stores: {"player|relation|lemma": [...]}
    for store in stores:
        for buckets in store.values():
            for relation, counts in buckets.items():
                for lemma in [w for w in counts if fixed_lemma(w) != w]:
                    counts[fixed_lemma(lemma)] += counts.pop(lemma)
    for store in example_stores:
        for key in list(store):
            who, relation, lemma = key.split("|", 2)
            if fixed_lemma(lemma) != lemma:
                store.setdefault(f"{who}|{relation}|{fixed_lemma(lemma)}", []).extend(store.pop(key))

def drop_not_her(found, examples, by_outlet, by_outlet_examples, rows):
    # each wrong attachment is found by its stored example, then taken off her
    # counts: overall, for its outlet and for its era
    dropped = 0
    for who, word, phrase in NOT_ABOUT_HER:
        date = next((r.get("date") or "" for r in rows if phrase in " ".join(r["body"].split())), None)
        hits = set()
        for store in [examples] + list(by_outlet_examples.values()):
            for key, exs in store.items():
                name, relation, lemma = key.split("|", 2)
                if name == who and word in lemma.split(" "):
                    hits.update((key, e.get("o") or "Fox News") for e in exs if phrase in e["t"])
                    exs[:] = [e for e in exs if phrase not in e["t"]]
        for key, outlet in hits:
            _, relation, lemma = key.split("|", 2)
            places = [found, by_outlet.get(outlet, {})]
            if date is not None:
                places.append(by_outlet.get("@pre" if date[:4] < "2024" else "@clark", {}))
            for store in places:
                counts = store.get(who, {}).get(relation)
                if counts and counts.get(lemma, 0) > 0:
                    counts[lemma] -= 1
                    if not counts[lemma]:
                        del counts[lemma]
            dropped += 1
    return dropped

for _counts, _examples, _by_outlet, _by_outlet_examples, _rows in (
        (found, examples, fox_by_outlet, {}, fox_rows),
        (other_found, other_examples, other_by_outlet, other_by_outlet_examples, other_rows)):
    rename_lemmas([_counts, *_by_outlet.values()], [_examples, *_by_outlet_examples.values()])
    print("attachments dropped by hand:", drop_not_her(_counts, _examples, _by_outlet, _by_outlet_examples, _rows))
other_found = defaultdict(lambda: defaultdict(Counter))
for _o, _store in other_by_outlet.items():
    if _o.startswith("@"):   # "@pre"/"@clark" are eras, not outlets
        continue
    for _p, _b in _store.items():
        for _r, _c in _b.items():
            other_found[_p][_r].update(_c)
print("comparison outlets:", {o: sum(sum(c.values()) for b in s.values() for c in b.values()) for o, s in other_by_outlet.items()})


## Participles

spaCy's small model tags passive participles as adjectives, so "she was violently fouled." Trying to correct. 

In [ ]:
TRUE_PARTICIPIAL_ADJ = {
    "excited", "worried", "concerned", "disappointed", "frustrated", "injured",
    "retired", "undefeated", "unfazed", "beloved", "talented", "gifted",
    "experienced", "respected", "celebrated", "grateful", "tired", "surprised",
    "determined", "committed", "qualified", "unranked", "ranked", "seeded",
}

def stems_of(word):
    # fouled -> foul, shoved -> shove, slammed -> slam
    base = word.rsplit("-", 1)[-1]
    if not base.endswith("ed"):
        return []
    out = {base[:-2], base[:-1]}
    if len(base) > 4 and base[-3] == base[-4]:
        out.add(base[:-3])
    return [w for w in out if len(w) > 2]


def reclassify_participles(found, examples, verbs=None):
    if verbs is None:
        verbs = set()
        for buckets in found.values():
            for relation in ("subject", "object"):
                verbs.update(buckets.get(relation, Counter()))

    moved = Counter()
    for player, buckets in found.items():
        describe = buckets.get("describe")
        if not describe:
            continue
        for word in list(describe):
            if word in TRUE_PARTICIPIAL_ADJ or not word.endswith("ed"):
                continue
            if re.search(r"-(hand|foot|side|sight|mind|will|heart)ed$", word):
                continue   # "right-handed" describes a person
            if not any(stem in verbs for stem in stems_of(word)):
                continue
            count = describe.pop(word)
            buckets.setdefault("object", Counter())[word] += count
            moved[word] += count
            old, new = f"{player}|describe|{word}", f"{player}|object|{word}"
            if old in examples:
                examples.setdefault(new, [])
                for line in examples.pop(old):
                    if line not in examples[new]:
                        examples[new].append(line)
    if moved:
        print("moved to 'done to her': "
              + ", ".join(f"{w} ({n})" for w, n in moved.most_common(8)))
    return found, examples

OTHER_VERBS = {w for b in other_found.values() for r in ("subject", "object") for w in b.get(r, Counter())}
found, examples = reclassify_participles(found, examples)
other_found, other_examples = reclassify_participles(other_found, other_examples)
for _o in list(other_by_outlet):
    other_by_outlet[_o], other_by_outlet_examples[_o] = reclassify_participles(other_by_outlet[_o], other_by_outlet_examples.get(_o, {}), verbs=OTHER_VERBS)


# "how she plays" is parsed broadly (ACT_NOUNS) with the noun kept beside the
# adjective; this is where it is narrowed to manner. "final season", "10th
# straight game" and "technical foul" are not how anyone plays.
SKIP_PLAY_ADJ = {"final", "entire", "past", "regular", "multiple", "single", "personal", "technical",
                 "consecutive", "straight", "overall", "collegiate", "professional", "pro", "senior",
                 "junior", "national", "own", "same", "first", "second", "third", "last", "next",
                 "early", "late", "whole", "full", "offensive", "defensive", "recent", "previous",
                 "upcoming", "opening", "big", "other", "many", "few", "several", "more", "most",
                 "much", "such", "only", "certain", "particular", "various", "new", "old", "former",
                 "current", "long", "short", "little", "high", "low", "double", "triple", "3-point",
                 "three-point", "free", "half", "midrange", "mid-range", "ridiculous", "viral",
                 "memorable", "historic", "illustrious", "storied", "career", "post", "home",
                 "road", "away", "playoff", "preseason", "postseason", "regular-season",
                 "all-star", "signature", "game-winning", "buzzer-beating", "back-to-back",
                 "usual", "similar", "inbound", "outside", "official", "cross", "complete", "well",
                 "foul", "possible", "potential", "actual", "real", "true", "sure", "able",
                 # adverbs and fragments the manner branch picks up ("of course", "since")
                 "course", "since", "potentially", "double-double", "triple-double", "also", "even",
                 # hedges, time words and the league's name are not how anyone plays
                 "maybe", "perhaps", "rather", "ago", "wnba", "wnba's", "lately", "non", "lone", "somehow",
                 "anyway", "instead", "apparently", "apparent", "seemingly", "arguably"}

# verbs of being on the receiving end: "took a hard foul", "absorbed a hit"
RECEIVE_VERBS = {"take", "receive", "endure", "absorb", "suffer", "face", "draw", "get", "use",
                 "survive", "withstand", "weather", "handle", "shrug", "brush", "avoid", "escape",
                 "dodge", "see", "experience", "encounter", "expect", "anticipate", "need", "want"}

# the neutral acts a play word may sit on: they no longer count as
# words themselves, but "fiery defense", "efficient shooting", "impressive play"
# are exactly the manner words the relation is for. Outcomes (season, night,
# game, career) stay out: "a strong season" is a result, not a way of playing.
PLAY_ANCHORS = {"play", "style", "defense", "defence", "offense", "offence", "effort", "approach",
                "tactic", "behavior", "behaviour", "shooting", "shot", "pass", "passing", "rebounding",
                "defending", "scoring", "playmaking", "footwork", "handle", "handles", "performance",
                # "a huge game from Boston", "great contributions", "a monster night": how she played
                "game", "night", "outing", "contribution", "showing"}

def settle_play(found, examples):
    for player, buckets in found.items():
        raw = buckets.pop("play", None)
        # the play phrases are the manner word with the act it is on ("cheap
        # shot", "efficient shooting"), from the same keys
        buckets.pop("play2", None)
        if not raw:
            continue
        kept, kept2 = Counter(), Counter()
        for key, count in raw.items():
            adj, noun, verb = (key.split("|") + ["", ""])[:3]
            # adjectives and adverbs only: the act noun itself ("foul",
            # "shove", "toughness") is an anchor, never a play word
            if (noun not in PLAY_NOUNS and noun not in PLAY_ANCHORS) or adj == noun \
                    or (adj in SKIP_PLAY_ADJ and adj != noun) or adj[:1].isdigit() or verb in RECEIVE_VERBS \
                    or verb in COMMENT_ACT:
                examples.pop(f"{player}|play|{key}", None)
                continue
            kept[adj] += count
            phrase = adj if " " in adj else f"{adj} {noun}"
            kept2[phrase] += count
            twin = examples.setdefault(f"{player}|play2|{phrase}", [])
            for line in examples.get(f"{player}|play|{key}", []):
                if line not in twin and len(twin) < MAX_EXAMPLES:
                    twin.append(line)
            old = examples.pop(f"{player}|play|{key}", [])
            new = examples.setdefault(f"{player}|play|{adj}", [])
            for line in old:
                if line not in new and len(new) < MAX_EXAMPLES:
                    new.append(line)
        if kept:
            buckets["play"] = kept
        if kept2:
            buckets["play2"] = kept2
    return found, examples

found, examples = settle_play(found, examples)
other_found, other_examples = settle_play(other_found, other_examples)
for _o in list(other_by_outlet):
    other_by_outlet[_o], other_by_outlet_examples[_o] = settle_play(other_by_outlet[_o], other_by_outlet_examples.get(_o, {}))

## Weighted log-odds



In [ ]:
# a word is letters, with an apostrophe or hyphen inside ("hip-check",
# "don't"); the parser's punctuation lemmas ("’", "'s", "'ve") are not words
WORDISH = re.compile(r"^[a-z][a-z'’-]*[a-z]$")

def contrast(subject, others, min_count=None):
    from sklearn.feature_extraction.text import CountVectorizer
    min_count = MIN_COUNT if min_count is None else min_count
    # a two-word phrase (the relations ending in 2) is scored as one token:
    # its space becomes "--" for the vectorizer (the log-odds helper strips
    # every punctuation mark but the hyphen) and back for the chart
    # hedges and time adverbs make empty partners ("hit actually", "ago
    # speak"), and "ing" is what is left of a bleeped f-ing
    FILLER = {"actually", "ago", "recently", "later", "previously", "eventually", "probably",
              "obviously", "definitely", "soon", "early", "once", "currently", "finally",
              "apparently", "reportedly", "certainly", "basically", "literally", "ing", "matter"}
    wordish = lambda w: all(WORDISH.match(part) for part in w.split(" ")) and \
        not (" " in w and any(part.removeprefix("not-") in FILLER for part in w.split(" ")))
    subject = Counter({w.replace(" ", "--"): n for w, n in subject.items() if wordish(w)})
    others = Counter({w.replace(" ", "--"): n for w, n in others.items() if wordish(w)})

    left = [" ".join([word] * n) for word, n in subject.items()]
    right = [" ".join([word] * n) for word, n in others.items()]
    if not left or not right:
        return {"clark": [], "others": []}
    # each lemma is one token already: the default pattern would split
    # "hip-check" into hip + check and "don't" into don + t
    vectorizer = CountVectorizer(decode_error="ignore", min_df=1, max_df=1.0,
                                 token_pattern=r"(?u)\S+")
    scores = fw.bayes_compare_language(left, right, ngram=1, prior=0.05, cv=vectorizer)
    total = subject + others
    kept = sorted([(w, z) for w, z in scores if total[w] >= min_count], key=lambda p: p[1])
    shape = lambda p: {"word": p[0].replace("--", " "), "z": round(p[1], 2),
                       "clark": subject.get(p[0], 0), "others": others.get(p[0], 0)}
    # every word above the floor goes in; the TOP_N strongest per side are
    # flagged for labels (and carry examples)
    mine = [shape(p) for p in kept[::-1] if p[1] > 0 and subject.get(p[0], 0) > 0]
    theirs = [shape(p) for p in kept if p[1] < 0 and others.get(p[0], 0) > 0]
    for side in (mine, theirs):
        for i, word in enumerate(side):
            word["top"] = i < TOP_N
    return {"clark": mine, "others": theirs}


def find_examples(store, who, relation, word):
    suffix = relation[3:] if relation in ("all", "all2") else ""
    parts = tuple(p + suffix for p in ("subject", "object", "describe", "play")) if suffix or relation == "all" else (relation,)
    out = []
    for part in parts:
        out.extend(store.get(f"{who}|{part}|{word}", []))
    return out[:MAX_EXAMPLES]

## Build the payload


In [ ]:
# each relation twice: its words, and (suffix 2) its two-word phrases
RELATIONS = ("subject", "object", "describe", "play", "all",
             "subject2", "object2", "describe2", "play2", "all2")

def bucket(store, name, relation):
    if relation not in ("all", "all2"):
        return store.get(name, {}).get(relation, Counter())
    suffix = relation[3:]
    merged = Counter()
    for part in ("subject", "object", "describe", "play"):
        merged.update(store.get(name, {}).get(part + suffix, Counter()))
    return merged

mentions = {name: sum(sum(c.values()) for c in b.values()) for name, b in found.items()}
cast = [n for n, total in sorted(mentions.items(), key=lambda kv: -kv[1])
        if total >= MIN_MENTIONS]
print(f"{len(cast)} players clear {MIN_MENTIONS} attachments ({len(found) - len(cast)} below)")

players = {}
for name in cast:
    entry = {"relations": {}, "counts": {}}
    for relation in RELATIONS:
        mine = bucket(found, name, relation)
        others = Counter()
        for peer in found:
            if peer != name:
                others.update(bucket(found, peer, relation))
        scored = contrast(mine, others, None)
        for side, owner in (("clark", name), ("others", None)):
            for word in scored[side]:
                if not word.get("top"):
                    continue
                if owner:
                    word["examples"] = find_examples(examples, owner, relation, word["word"])
                else:
                    # pooled column: examples from whichever peer uses the word most
                    best, count = None, 0
                    for peer, buckets in found.items():
                        if peer == name:
                            continue
                        got = buckets.get(relation, Counter()).get(word["word"], 0)
                        if got > count:
                            best, count = peer, got
                    word["examples"] = examples.get(
                        f"{best}|{relation}|{word['word']}", []) if best else []
                    word["who"] = best
        entry["relations"][relation] = scored
        entry["counts"][relation] = {"clark": sum(mine.values()),
                                     "others": sum(others.values())}

        theirs = bucket(other_found, name, relation)
        # play attachments are rare (an adjective on a foul or a shot): a floor
        # of 10 a side lets Cunningham (14 / 10) and Reese (10 / 18) in
        # no floor beyond one word a side: the log-odds prior already
        # keeps a thin comparison's scores near zero, and the chart says how thin it is
        floor = 1
        if relation == "play":
            print(f"  play · {name}: Fox {sum(mine.values())} · other outlets {sum(theirs.values())}")
        if sum(mine.values()) >= floor and sum(theirs.values()) >= floor:
            versus = contrast(mine, theirs, None)
            for word in versus["clark"]:
                if word.get("top"):
                    word["examples"] = find_examples(examples, name, relation, word["word"])
            for word in versus["others"]:
                if word.get("top"):
                    word["examples"] = find_examples(other_examples, name, relation, word["word"])
            entry.setdefault("vsOutlets", {})[relation] = versus
            entry.setdefault("vsCounts", {})[relation] = {
                "clark": sum(mine.values()), "others": sum(theirs.values())}
        # the same contrast against each comparison outlet on its own
        for outlet_name, store_o in sorted(other_by_outlet.items()):
            if outlet_name.startswith("@"):
                continue
            theirs_o = bucket(store_o, name, relation)
            if sum(mine.values()) >= floor and sum(theirs_o.values()) >= floor:
                versus_o = contrast(mine, theirs_o, None)
                for word in versus_o["clark"]:
                    if word.get("top"):
                        word["examples"] = find_examples(examples, name, relation, word["word"])
                for word in versus_o["others"]:
                    if word.get("top"):
                        word["examples"] = find_examples(other_by_outlet_examples.get(outlet_name, {}), name, relation, word["word"])
                entry.setdefault("vsByOutlet", {}).setdefault(relation, {})[outlet_name] = {
                    "scored": versus_o,
                    "counts": {"clark": sum(mine.values()), "others": sum(theirs_o.values())}}
        # and against each pooled group (sports, news): the
        # members' stores summed, examples drawn from whichever member has one
        from outlets import GROUPS as _GROUPS
        for group_name, members in _GROUPS.items():
            if group_name == "all outlets":
                continue
            pooled_g = Counter()
            for m in members:
                if m in other_by_outlet:
                    pooled_g.update(bucket(other_by_outlet[m], name, relation))
            if sum(mine.values()) >= floor and sum(pooled_g.values()) >= floor:
                versus_g = contrast(mine, pooled_g, None)
                for word in versus_g["clark"]:
                    if word.get("top"):
                        word["examples"] = find_examples(examples, name, relation, word["word"])
                for word in versus_g["others"]:
                    if word.get("top"):
                        got = []
                        for m in members:
                            got += find_examples(other_by_outlet_examples.get(m, {}), name, relation, word["word"])
                        word["examples"] = got[:MAX_EXAMPLES]
                entry.setdefault("vsByOutlet", {}).setdefault(relation, {})[group_name] = {
                    "scored": versus_g,
                    "counts": {"clark": sum(mine.values()), "others": sum(pooled_g.values())}}
    players[name] = entry

In [ ]:
PAIRS = [("Caitlin Clark", "Angel Reese")]
pairs = {}
for left, right in PAIRS:
    if left not in found or right not in found:
        continue
    block = {"relations": {}, "counts": {}}
    for relation in RELATIONS:
        a = bucket(found, left, relation)
        b = bucket(found, right, relation)
        if sum(a.values()) < 25 or sum(b.values()) < 25:
            continue
        scored = contrast(a, b)
        for word in scored["clark"]:
            if word.get("top"):
                word["examples"] = find_examples(examples, left, relation, word["word"])
        for word in scored["others"]:
            if word.get("top"):
                word["examples"] = find_examples(examples, right, relation, word["word"])
        block["relations"][relation] = scored
        block["counts"][relation] = {"clark": sum(a.values()), "others": sum(b.values())}
    pairs[f"{left} vs {right}"] = {**block, "left": left, "right": right}

payload = {"order": cast, "mentions": {n: mentions[n] for n in cast},
           "pairs": pairs,
           "players": players,
           "hasOutletComparison": bool(other_found),
           "source": "article bodies",
           "nDocs": len(fox_rows)}

#### Overwrites `src/data/player_words.json`

In [ ]:
# slim copy for the site: one example survives per word (hover), the rest dropped.
# An audited example that was judged wrong (notebooks/example_audit.py: the
# pronoun is somebody else, or the word is not about her play) is skipped for
# the next candidate; the candidates that need auditing are written out.
from outlets import GROUPS as SITE_GROUPS   # the pooled groups whose examples ship on the site
AUDIT_FILE = CACHE / "example_audit.json"
AUDIT = json.loads(AUDIT_FILE.read_text()) if AUDIT_FILE.exists() else {}
AUDIT_THRESH = 0.85          # a pronoun below this confidence goes for audit
def audit_id(who, rel, ex):
    return hashlib.md5(f"{who}|{rel}|{ex.get('w')}|{ex['t']}".encode()).hexdigest()[:16]
def audit_ok(who, rel, ex):
    return AUDIT.get(audit_id(who, rel, ex), {}).get("verdict") not in ("wrong", "no")
def needs_audit(who, rel, ex):
    if audit_id(who, rel, ex) in AUDIT:
        return False
    if ex.get("pr") and (ex.get("p") is None or ex["p"] < AUDIT_THRESH):
        return True
    return rel == "play"
candidates = {}
def collect(who, rel, exs, kind):
    for ex in exs[:MAX_EXAMPLES]:
        if isinstance(ex, dict) and needs_audit(who, rel, ex):
            candidates.setdefault(audit_id(who, rel, ex), {
                "id": audit_id(who, rel, ex), "player": who, "relation": rel, "word": ex.get("w"),
                "pronoun": ex.get("pr"), "confidence": ex.get("p"), "outlet": ex.get("o"), "kind": kind, "text": ex["t"]})
slim = json.loads(json.dumps(payload))
# the player-versus-other-players block is not on the site; keep only its labelled words
for block in slim["players"].values():
    for scored in block["relations"].values():
        for side in ("clark", "others"):
            scored[side] = [w for w in scored[side] if w.get("top")]
for who_, block in list(slim["players"].items()) + list(slim["pairs"].items()):
    for group in ("relations", "vsOutlets"):
        for rel_, scored in (block.get(group) or {}).items():
            for side in ("clark", "others"):
                for word in scored[side]:
                    examples_here = word.pop("examples", None) or []
                    owner = word.get("who") or (who_ if side == "clark" or group == "vsOutlets" else None)
                    if owner and "|" not in who_:
                        collect(owner, rel_, examples_here, group)
                        examples_here = [e for e in examples_here if not isinstance(e, dict) or audit_ok(owner, rel_, e)]
                    if examples_here:
                        first = examples_here[0]
                        if isinstance(first, dict):
                            word["eg"] = first["t"]
                            word["egw"] = first["w"]
                            if first.get("a") is not None:
                                word["ega"] = first["a"]   # where the word starts in it
                            if first.get("o"):
                                word["ego"] = first["o"]   # the outlet the example is from
                        else:
                            word["eg"] = first

for who_, block in slim["players"].items():
    for rel_, rel_map in (block.get("vsByOutlet") or {}).items():
        for group_, cmp in rel_map.items():
            for side in ("clark", "others"):
                kept_words = []
                for w in cmp["scored"][side][:80]:
                    w = dict(w); ex = w.pop("examples", None) or []
                    if group_ in SITE_GROUPS:       # the pooled groups ship; single outlets are dev-only
                        collect(who_, rel_, ex, "vsByOutlet")
                    ex = [e for e in ex if audit_ok(who_, rel_, e)]
                    if ex and isinstance(ex[0], dict):
                        w["eg"], w["egw"], w["ego"] = ex[0]["t"], ex[0]["w"], ex[0].get("o")
                        if ex[0].get("a") is not None:
                            w["ega"] = ex[0]["a"]
                    kept_words.append(w)
                cmp["scored"][side] = kept_words

# the single-outlet comparisons leave the bundle for one file per player
# (notebooks/site_data.py); the group pills stay
from site_data import write_player_words
write_player_words(slim)
(CACHE / "audit_candidates.jsonl").write_text("\n".join(json.dumps(c, ensure_ascii=False) for c in candidates.values()))
print(f"{len(candidates):,} examples to audit ({sum(1 for c in candidates.values() if c['pronoun'])} pronouns, "
      f"{sum(1 for c in candidates.values() if c['relation'] == 'play')} play); {len(AUDIT):,} verdicts on file")

## Preview

In [ ]:
new = json.loads(OUT.read_text())
preview = []
for name in new["order"]:
    rels = new["players"][name]["relations"]
    preview.append({
        "player": name,
        "mentions": new["mentions"][name],
        "she does": ", ".join(w["word"] for w in rels["subject"]["clark"][:5]),
        "done to her": ", ".join(w["word"] for w in rels["object"]["clark"][:5]),
        "described as": ", ".join(w["word"] for w in rels["describe"]["clark"][:5]),
        "how she plays": ", ".join(w["word"] for w in rels.get("play", {}).get("clark", [])[:5]),
    })
pd.DataFrame(preview)

## Spot check

In [ ]:
import random

player = random.choice(list(payload["players"]))
rel = random.choice(["subject", "object", "describe"])
side = payload["players"][player]["relations"].get(rel, {}).get("clark", [])
print(f"— {player} · {rel} ('she does' / 'done to her' / 'described as')")
for w in random.sample(side, min(3, len(side))):
    print(f"   {w['word']} ({w.get('count', '?')})")
    for ex in (w.get("examples") or [])[:2]:
        text = ex["t"] if isinstance(ex, dict) else ex
        print(f"      “{str(text)[:100]}”")